In [7]:
import os 
import subprocess

In [47]:
import json
import pandas as pd
df = pd.read_csv("latin_square_design.csv")
index = 0
p1 = df.iloc[0]


#研究被験者の取得
participant = f"P{index+1}"
taskSet = p1["Task Set 1"]
condition_num = int(p1["Condition 1_ConditionNum"])




#ArrangeDataを取得する
with open(f"../InteractiveSmartHome/Assets/EXPERIMENT/ArrangeData/PreTaskArrangement{taskSet}.json", "r", encoding="utf-8") as f:
    arrangeData = json.loads(f.read())

with open(f"../InteractiveSmartHome/Assets/EXPERIMENT/VOICE_LOG/{participant}/{condition_num}_{taskSet}.json", "r", encoding="utf-8") as f:
    voice_log = json.loads(f.read())
    
voi

P1
A
1


Participant    Participant 1
Condition 1           空間参照だけ
Task Set 1                 A
Condition 2        ポインティングだけ
Task Set 2                 B
Name: 0, dtype: object

In [34]:
with open(f"../LLMServer/ExperimentData/RESULTS/P1.json", "r", encoding="utf-8") as f:
    result_data = json.loads(f.read())


result_data[-1]

{'user_prompt': '俺より高いところにある電気を全部 つけ',
 'filterAgent': {'output_tool_selection': {'filter_type': 'direction',
   'params': {'direction': 'Up', 'order': 'high', 'range': 0.0},
   'reasoning': '『俺より高いところにある』という身体基準の方向参照が含まれているため、directionフィルタを選択し、方向はUp、順序はhighとした。'},
  'devices': [{'id': '34bf7b7e-946d-4afc-a929-58a4cbebd694',
    'name': '天井ライト1',
    'position': {'x': 2.86795282, 'y': 2.30608034, 'z': 3.57684016},
    'distance_from_user': 5.13195324,
    'eye_centrality_score': 3.40282347e+38},
   {'id': '01ed37d3-f6df-46c6-ad5a-595d04ec9eed',
    'name': '天井ライト2',
    'position': {'x': 2.43614674, 'y': 2.30608034, 'z': 1.64449883},
    'distance_from_user': 3.73593283,
    'eye_centrality_score': 3.40282347e+38},
   {'id': '6798cacb-70ee-451b-b218-d15b9a544e5d',
    'name': '天井ライト3',
    'position': {'x': 1.991692, 'y': 2.30608034, 'z': -0.3444472},
    'distance_from_user': 3.06651068,
    'eye_centrality_score': 3.40282347e+38},
   {'id': 'bce898b8-d5d6-45ac-b337-a2820b37c357',
   

In [24]:
outputData[0]["task_id"]="3a642a22-d8be-466d-ae3d-8ee4ae7e730f"

In [2]:
def evaluate_all_predictions(output_data_list, arrange_data_list):
    results = []

    # arrange_data を dict に変換して素早くアクセス
    arrange_map = {
        entry["device_arrange_id"]: [dev["deviceId"] for dev in entry["devices"]]
        for entry in arrange_data_list
    }

    for output in output_data_list:
        task_id = output.get("task_id")
        predicted_ids = [d["id"] for d in output.get("selected_devices", [])]
        ground_truth_ids = arrange_map.get(task_id)

        if ground_truth_ids is None:
            print(f"[⚠️ Warning] Ground truth not found for task_id: {task_id}")
            continue

        gt_set = set(ground_truth_ids)
        pred_set = set(predicted_ids)

        true_positives = gt_set & pred_set
        false_positives = pred_set - gt_set
        false_negatives = gt_set - pred_set

        precision = len(true_positives) / len(pred_set) if pred_set else 0
        recall = len(true_positives) / len(gt_set) if gt_set else 0

        results.append({
            "task_id": task_id,
            "ground_truth": ground_truth_ids,
            "predicted": predicted_ids,
            "true_positives": list(true_positives),
            "false_positives": list(false_positives),
            "false_negatives": list(false_negatives),
            "precision": precision,
            "recall": recall
        })

    return results


In [26]:
results = evaluate_all_predictions(outputData, arrangeData)
import pprint
pprint.pprint(results)


[{'false_negatives': ['a65c2e1d-fbe8-4bf5-a49b-0ea4ad1db5c3',
                      'c364e75e-13c7-4a1f-a48b-c431eff17e45',
                      'bce898b8-d5d6-45ac-b337-a2820b37c357',
                      '81406953-308f-4a7d-bc6b-340c969cbc29',
                      '36b0460e-36d2-4f5a-88a5-5174657039f1',
                      '1c33a40f-f48c-49f5-8383-d1c006559204'],
  'false_positives': ['3f613e25-0fa0-4780-9896-adcdf01474c2',
                      '5267d97b-9929-4288-a957-15fd46185712'],
  'ground_truth': ['a65c2e1d-fbe8-4bf5-a49b-0ea4ad1db5c3',
                   'c364e75e-13c7-4a1f-a48b-c431eff17e45',
                   '81406953-308f-4a7d-bc6b-340c969cbc29',
                   'bce898b8-d5d6-45ac-b337-a2820b37c357',
                   '36b0460e-36d2-4f5a-88a5-5174657039f1',
                   '1c33a40f-f48c-49f5-8383-d1c006559204'],
  'precision': 0.0,
  'predicted': ['5267d97b-9929-4288-a957-15fd46185712',
                '3f613e25-0fa0-4780-9896-adcdf01474c2'],
  'recall': 0.

In [12]:
import pandas as pd
import re

# CSVの読み込み
condition = pd.read_csv("latin_square_design.csv")

# 変換用マッピング
replace_map = {
    "FD+SR": "空間参照だけ",
    "Pointing": "ポインティングだけ",
    "Pointing+SR": "Pointing+空間参照",
    "Label": "ラベル"
}

# Condition列だけを対象に処理
for col in condition.columns:
    if "Condition" in col:
        # ()の中の数字を取り出し、新しい列に保存
        condition[col + "_ConditionNum"] = condition[col].str.extract(r"\((\d+)\)").astype(float)

        # ()を除いた本体部分を変換
        condition[col] = condition[col].str.replace(r"\(\d+\)", "", regex=True).str.strip()
        condition[col] = condition[col].replace(replace_map)

# 結果を表示
condition.to_csv("./latin_square_design.csv", index=False)